# Directional sweeps — signal construction

Testing the proposed strategy: **golden sweep, ISO, premium >= $1M, delta 0.30-0.70,
confirmed by the other sweep activity on that ticker.**

This notebook builds and stress-tests the *signal*. It does not yet measure returns —
see the last section for what that needs. The purpose is to settle which version of
the rule is worth pricing, because two of the five filters behave differently than
they look.

Everything reads a local parquet cache, so re-running costs nothing. Companion to
[`research/quantdata/explore_api.ipynb`](../quantdata/explore_api.ipynb), which maps
the feed this is built on.

## Where the spec already stands

Three of the five filters are not new — they are the existing sweep log's screen:

| Rule | Status |
|---|---|
| Quant Data golden sweep | `tradeConsolidationTypes=["SWEEP"]` — already the screen |
| Type = ISO | `tradeTypes=["ISO"]` — already the screen |
| Premium >= $1M | already the floor; the log's smallest entry is $1.03M |
| Delta 0.30-0.70 | already `DELTA_MIN/DELTA_MAX` in `build_log.py`, as `delta_in_band` |
| **Confirm via other sweeps on the ticker** | **new — the part worth testing** |

So the work is in the fifth rule, plus one correction to the trigger itself that the
others depend on.

## 1. Data

`fetch_flow.py` caches every ISO sweep print at a **$100k** floor — deliberately far
below the $1M trigger. The confirmation signal needs the small prints to count: at a
$1M floor a session holds ~13 prints across ~11 tickers, so nearly every ticker has
exactly one and there is nothing to confirm against.

```bash
python research/sweep_events/fetch_flow.py 2026-06-01 2026-09-21   # skips cached days
```

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "research" / "sweep_events"))
import directional as D  # noqa: E402

pd.set_option("display.width", 150, "display.max_columns", 40)

# ---- knobs: change these and re-run from here down ------------------------
TRIGGER_PREMIUM = 1_000_000
DELTA_BAND      = (0.30, 0.70)
LOOKBACK        = 0        # extra sessions of context; 0 = trigger's own session
EXCLUDE_FUNDS   = True     # ETFs and indices — see section 2
EXCLUDE_TICKERS = ("SNDK",)  # by name
MAX_COST        = None     # or a $/contract cap — see section 2b
# ---------------------------------------------------------------------------

flow = D.load()
print(f"{len(flow):,} ISO sweep prints >= $100k")
print(f"{flow.ticker.nunique()} tickers, {flow.session.nunique()} sessions, "
      f"{flow.session.min()} .. {flow.session.max()}")

25,610 ISO sweep prints >= $100k
853 tickers, 78 sessions, 2026-06-01 .. 2026-09-21


## 2. The trigger — and the correction it needs first

Before counting anything, direction has to be right. §1 of `strategy.md` takes
direction from the sweep's call/put. The feed says that is wrong for most prints:
**direction is `contractType` x `tradeSideCode`.** Buying calls or selling puts is
bullish; selling calls or buying puts is bearish.

If this is skipped, every downstream filter is conditioned on an inverted sign, and
the confirmation test below measures agreement between two wrong labels.

In [2]:
naive = flow["contractType"].map({"CALL": "BULL", "PUT": "BEAR"})
wrong = (naive != flow["direction"]) & flow["direction"].notna()

print(f"prints where call/put ALONE gives the wrong direction: "
      f"{wrong.sum():,} of {flow['direction'].notna().sum():,} ({wrong.mean():.0%})")
print()
print(pd.crosstab(flow["contractType"], flow["tradeSideCode"]))

prints where call/put ALONE gives the wrong direction: 12,101 of 25,220 (47%)

tradeSideCode  ABOVE_ASK   ASK  BELOW_BID   BID  MID_MARKET
contractType                                               
CALL                1063  6077       1538  5106         223
PUT                  822  5157       1358  4099         167


`MID_MARKET` prints have no aggressor, so they have no direction. They are dropped
rather than defaulted to a side — a coin flip dressed as a signal is worse than a
missing one.

Now the trigger population:

In [3]:
trig = D.triggers(flow, premium=TRIGGER_PREMIUM, delta_band=DELTA_BAND,
                  exclude_funds=EXCLUDE_FUNDS, exclude_tickers=EXCLUDE_TICKERS,
                  max_contract_cost=MAX_COST)

print(f"TRIGGERS: {len(trig)} prints over {flow.session.nunique()} sessions "
      f"= {len(trig) / flow.session.nunique():.1f} per session")
print(f"{trig.ticker.nunique()} distinct tickers")
print(f"direction: {trig.direction.value_counts().to_dict()}")
print()
print("survivorship through each filter:")
step = flow[flow.premium >= TRIGGER_PREMIUM]
print(f"  ISO + SWEEP + >=$100k          {len(flow):>6,}")
print(f"  ... premium >= ${TRIGGER_PREMIUM/1e6:.0f}M            {len(step):>6,}")
print(f"  ... delta {DELTA_BAND[0]}-{DELTA_BAND[1]}              "
      f"{len(step[step.absdelta.between(*DELTA_BAND)]):>6,}")
print(f"  ... has an aggressor side      "
      f"{len(step[step.absdelta.between(*DELTA_BAND) & step.direction.notna()]):>6,}")
print(f"  ... single names only          "
      f"{len(D.triggers(flow, premium=TRIGGER_PREMIUM, delta_band=DELTA_BAND, exclude_tickers=())):>6,}")
print(f"  ... after ticker/cost screen   {len(trig):>6,}")

TRIGGERS: 441 prints over 78 sessions = 5.7 per session
82 distinct tickers
direction: {'BULL': 225, 'BEAR': 216}

survivorship through each filter:
  ISO + SWEEP + >=$100k          25,610
  ... premium >= $1M             1,004
  ... delta 0.3-0.7                 645
  ... has an aggressor side         642
  ... single names only             471
  ... after ticker/cost screen      441


Roughly 8 triggers a session — about 2,000 a year, against the 118 in the hand-built
log. Sample size stops being the binding constraint.

Note the direction split is close to even. The trigger carries no directional tilt of
its own, which is what makes the confirmation question worth asking.

### ETFs and indices are excluded

37% of all sweep prints (9,568 of 25,610) are on ETFs or indices. They are dropped:
a sweep on QQQ, SPY, SMH or a leveraged product like BITX is portfolio hedging or
index expression, not a view on a company — the wrong population for a signal whose
premise is that someone knows something specific.

It costs about a quarter of the triggers. It does **not** rescue the raw count, which
is the point of the next section — MU, NVDA and TSLA sweep constantly too, so the
liquidity confound survives the cut.

In [4]:
funds = D.is_fund(flow)
print(f"fund prints: {int(funds.sum()):,} of {len(flow):,} ({funds.mean():.0%})")
print()
for label, drop in [("with ETFs/indices", False), ("single names only", True)]:
    t = D.triggers(flow, premium=TRIGGER_PREMIUM, delta_band=DELTA_BAND, exclude_funds=drop)
    c = D.confirm(flow, t)
    heavy = c[c.prior_n > 10].aligned.mean()
    print(f"  {label:20} {len(t):4} triggers  {t.ticker.nunique():3} tickers  "
          f"mean aligned {c.aligned.mean():+.2f}  heavy-tier aligned {heavy:+.2f}")

fund prints: 9,655 of 25,610 (38%)



  with ETFs/indices     612 triggers  105 tickers  mean aligned +0.16  heavy-tier aligned +0.07


  single names only     441 triggers   82 tickers  mean aligned +0.18  heavy-tier aligned +0.06


### 2b. Affordability — the screen that should replace the name

SNDK trades near $1,640, so one contract costs a median **$15,544** — 1.6x the next
worst name and unaffordable at any realistic position cap. It is excluded.

But excluding it by name treats the symptom. ARM ($9.7k) and MU ($9.0k) are barely
better, and **MU alone is 65 of the triggers** against SNDK's 30. Screening on what a
contract actually costs states the real constraint and keeps working as new expensive
names appear:

In [5]:
cost = (trig.optionPrice * 100)
by_cost = (D.triggers(flow, exclude_tickers=())
           .groupby("ticker")
           .agg(triggers=("premium", "size"), med_spot=("stockPrice", "median"),
                contract_cost=("optionPrice", lambda s: round(s.median() * 100)))
           .query("triggers >= 4").sort_values("contract_cost", ascending=False))
print("cost of one contract at the trigger's own price (median):")
print(by_cost.head(8).round(0).to_string())
print()
for label, kw in [("no screen",            dict(exclude_tickers=())),
                  ("SNDK removed by name", dict()),
                  ("cost cap $5k",         dict(exclude_tickers=(), max_contract_cost=5_000)),
                  ("cost cap $2k",         dict(exclude_tickers=(), max_contract_cost=2_000))]:
    t = D.triggers(flow, premium=TRIGGER_PREMIUM, delta_band=DELTA_BAND, **kw)
    c = D.confirm(flow, t)
    print(f"  {label:22} {len(t):4} triggers  {t.ticker.nunique():3} tickers  "
          f"mean aligned {c.aligned.mean():+.2f}")

cost of one contract at the trigger's own price (median):
        triggers  med_spot  contract_cost
ticker                                   
SNDK          30    1641.0          15544
ARM            7     328.0           9679
MU            65    1000.0           9000
COHR           6     381.0           7304
NBIS          16     269.0           6388
BE            17     260.0           5585
DELL           4     505.0           4860
MRVL          10     266.0           4413



  no screen               471 triggers   83 tickers  mean aligned +0.18


  SNDK removed by name    441 triggers   82 tickers  mean aligned +0.18


  cost cap $5k            317 triggers   76 tickers  mean aligned +0.25


  cost cap $2k            172 triggers   57 tickers  mean aligned +0.25


Removing SNDK by name changes the signal not at all (+0.18 either way) — it is 30
triggers out of 441. A **$5k cost cap lifts mean alignment to +0.25** while keeping
317 triggers, because it also removes MU, ARM, COHR and the rest of the expensive
tail in one rule.

Treat that lift as suggestive, not established. Cheaper contracts mean lower-priced
underlyings, which may simply be a different kind of company with different flow — the
cap could be picking up a size effect rather than improving the signal. It is one more
knob that has to clear an out-of-sample test before it earns a place.

## 3. The confirmation rule, exactly as proposed

"Verify the golden sweep by looking at the value count of sweeps for that ticker."

Computed **point-in-time** — only prints that had already happened when the trigger
fired. Counting a ticker's whole-day or whole-sample activity would be lookahead: it
selects tickers that *turned out* to be heavily swept.

In [6]:
conf = D.confirm(flow, trig, lookback_sessions=LOOKBACK)

print("prior sweeps on the same ticker, before the trigger printed:")
print(conf.prior_n.describe()[["mean", "50%", "75%", "max"]].round(1).to_string())
print()
tiers = pd.cut(conf.prior_n, [-1, 0, 2, 5, 10, 10_000],
               labels=["0", "1-2", "3-5", "6-10", "11+"])
print(tiers.value_counts().sort_index().rename("triggers").to_string())
print()
print(f"triggers with NO prior context at all: {(conf.prior_n == 0).sum()} "
      f"of {len(conf)} ({(conf.prior_n == 0).mean():.0%})")

prior sweeps on the same ticker, before the trigger printed:
mean     5.3
50%      3.0
75%      7.0
max     53.0

prior_n
0       126
1-2      90
3-5      83
6-10     70
11+      72

triggers with NO prior context at all: 126 of 441 (29%)


### Why the raw count does not do what it looks like it does

A quarter of triggers have nothing to confirm against, so the rule is silent on them.
That is survivable. The real problem is what the count *is*:

In [7]:
by_ticker = (conf.groupby("ticker")
             .agg(triggers=("prior_n", "size"), mean_prior=("prior_n", "mean"),
                  is_etf=("isETF", "first"))
             .query("triggers >= 5").sort_values("mean_prior", ascending=False))

print("tickers by mean prior-sweep count (>=5 triggers):")
print(by_ticker.head(10).round(1).to_string())
print()
# The ETF correlation has to be measured BEFORE the exclusion, or it is NaN by
# construction. It is the cleanest evidence that the count tracks liquidity.
everything = D.confirm(flow, D.triggers(flow, premium=TRIGGER_PREMIUM,
                                        delta_band=DELTA_BAND, exclude_funds=False))
print(f"on the unfiltered trigger set, correlation prior_n vs isETF : "
      f"{everything[['prior_n', 'isETF']].astype(float).corr().iloc[0, 1]:+.3f}")
print()
print("...and the confound SURVIVES the exclusion — these are all single names:")
print(f"  median prior_n across triggers : {conf.prior_n.median():.0f}")
print(f"  MU / TSLA / NVDA mean prior_n  : "
      f"{conf[conf.ticker.isin(['MU','TSLA','NVDA'])].prior_n.mean():.1f}")

tickers by mean prior-sweep count (>=5 triggers):
        triggers  mean_prior  is_etf
ticker                              
MU            65        11.3   False
TSLA          26        10.7   False
META          18         8.7   False
SPCX          23         8.3   False
NVDA          48         8.2   False
INTC          17         7.8   False
AMD           13         3.8   False
MSFT           8         3.4   False
PLTR           9         3.1   False
MRVL          10         2.7   False



on the unfiltered trigger set, correlation prior_n vs isETF : +0.411

...and the confound SURVIVES the exclusion — these are all single names:
  median prior_n across triggers : 3
  MU / TSLA / NVDA mean prior_n  : 10.2


**The count is a liquidity measure, not a conviction measure.** Before the fund
exclusion the count correlated +0.41 with merely *being an ETF* — QQQ averaged ~29
prior sweeps and SPY ~19, against a median of 3.

Dropping ETFs does not fix it. MU, TSLA and NVDA average 8-11 prior sweeps on their
own: they sweep constantly regardless of what they are about to do. A threshold like
"require 5+ prior sweeps" still selects the busiest handful of single names rather
than institutional agreement.

And the count is nearly blind to direction, which is the thing being confirmed:

In [8]:
ctx = conf[conf.prior_n > 0]
agree = [(ctx_row.direction == flow[(flow.ticker == ctx_row.ticker)
                                    & (flow.session == ctx_row.session)
                                    & (flow.ts < ctx_row.ts)].direction).sum()
         for ctx_row in ctx.itertuples()]
disagree = [len(flow[(flow.ticker == r.ticker) & (flow.session == r.session)
                     & (flow.ts < r.ts) & flow.direction.notna()]) - a
            for r, a in zip(ctx.itertuples(), agree)]

cmp = pd.DataFrame({"agree": agree, "disagree": disagree})
print(f"of {len(cmp)} triggers with context:")
print(f"  mean prior sweeps AGREEING with the trigger   : {cmp.agree.mean():.1f}")
print(f"  mean prior sweeps DISAGREEING                 : {cmp.disagree.mean():.1f}")
print(f"  triggers where agreement outweighs disagreement: "
      f"{(cmp.agree > cmp.disagree).sum()} ({(cmp.agree > cmp.disagree).mean():.0%})")

of 315 triggers with context:
  mean prior sweeps AGREEING with the trigger   : 3.8
  mean prior sweeps DISAGREEING                 : 3.5
  triggers where agreement outweighs disagreement: 159 (50%)


53% — a coin flip. "This ticker has lots of sweeps" carries almost no information
about *which way* those sweeps lean. The rule as specified would filter the signal
set down without filtering it toward anything.

## 4. The reformulation — measure one-sidedness, not volume

The instinct is sound; the statistic is wrong. What confirmation should mean is that
the other flow **leans the same way**, and that is a ratio, not a count:

```
conviction = (bullish premium - bearish premium) / total premium      in [-1, +1]
aligned    = conviction, signed toward the trigger's own direction
```

Scale-free by construction, so a $2M ticker and a $200M ticker are comparable, and
one enormous print cannot manufacture agreement on its own.

In [9]:
print(f"correlation, conviction vs prior_n : "
      f"{conf[['conviction', 'prior_n']].corr().iloc[0, 1]:+.3f}   (raw count was +0.41 vs isETF)")
print()
print("alignment — positive means prior flow leans the trigger's way:")
print(conf.aligned.describe()[["count", "mean", "25%", "50%", "75%"]].round(2).to_string())
print()
print(f"  leaning the same way      : {(conf.aligned > 0).sum()} of {conf.aligned.notna().sum()}")
print(f"  strongly aligned  (>+0.5) : {(conf.aligned > 0.5).sum()}")
print(f"  strongly opposed  (<-0.5) : {(conf.aligned < -0.5).sum()}")

correlation, conviction vs prior_n : +0.011   (raw count was +0.41 vs isETF)

alignment — positive means prior flow leans the trigger's way:
count    315.00
mean       0.18
25%       -0.35
50%        0.26
75%        0.83

  leaning the same way      : 192 of 315
  strongly aligned  (>+0.5) : 116
  strongly opposed  (<-0.5) : 61


Two results worth keeping.

**The conviction ratio is not a size proxy** — it correlates -0.03 with the prior
count, where the raw count correlated +0.41 with merely being an ETF. It measures
something the count does not.

**Strong alignment outnumbers strong opposition roughly two to one** (160 vs 84).
That asymmetry is the actual candidate signal. It is not proof of edge — no returns
have been measured — but it is a real, directional, non-liquidity feature, which the
raw count was not.

And the count is actively unhelpful as a gate:

In [10]:
conf["tier"] = pd.cut(conf.prior_n, [-1, 0, 2, 10, 10_000],
                      labels=["none", "thin 1-2", "mid 3-10", "heavy 11+"])
print(conf.groupby("tier", observed=True)
      .agg(triggers=("aligned", "size"), mean_aligned=("aligned", "mean"),
           frac_leaning_same=("aligned", lambda s: (s > 0).mean()))
      .round(2).to_string())

           triggers  mean_aligned  frac_leaning_same
tier                                                
none            126           NaN               0.00
thin 1-2         90          0.23               0.62
mid 3-10        153          0.20               0.63
heavy 11+        72          0.06               0.54


The **heavy** tier — exactly the tickers a count threshold would select — has the
*weakest* alignment (0.07 against 0.21 for the mid tier). Filtering on "lots of
sweeps" would preferentially keep the signals with the least directional conviction.
That is the rule as proposed working backwards.

## 5. Knobs

Re-run section 1 with these changed. Each is a real fork in the spec, not a tuning
dial — decide them on reasoning, then test, rather than searching for the best number
on 78 sessions.

In [11]:
grid = []
for prem in (500_000, 1_000_000, 2_000_000):
    for band in ((0.30, 0.70), (0.40, 0.60), (0.20, 0.80)):
        t = D.triggers(flow, premium=prem, delta_band=band)
        c = D.confirm(flow, t, lookback_sessions=0)
        grid.append({"premium": f"${prem/1e6:.1f}M", "delta": f"{band[0]}-{band[1]}",
                     "triggers": len(t), "per_session": round(len(t) / flow.session.nunique(), 1),
                     "tickers": t.ticker.nunique(),
                     "mean_aligned": round(c.aligned.mean(), 2),
                     "strong_align": int((c.aligned > 0.5).sum())})
pd.DataFrame(grid)

,premium,delta,triggers,per_session,tickers,mean_aligned,strong_align
0,$0.5M,0.3-0.7,1377,17.7,163,0.13,351
1,$0.5M,0.4-0.6,870,11.2,132,0.14,219
2,$0.5M,0.2-0.8,1647,21.1,184,0.13,430
3,$1.0M,0.3-0.7,441,5.7,82,0.18,116
4,$1.0M,0.4-0.6,291,3.7,69,0.18,73
5,$1.0M,0.2-0.8,540,6.9,92,0.19,150
6,$2.0M,0.3-0.7,132,1.7,42,0.16,36
7,$2.0M,0.4-0.6,87,1.1,33,0.16,24
8,$2.0M,0.2-0.8,163,2.1,46,0.18,46


## 6. What this still needs before it is a backtest

Nothing above measures a return. Signal construction is settled; pricing is not.

| Need | Where it comes from | State |
|---|---|---|
| Forward underlying path | Alpaca bars, split-adjusted | available, unwired here |
| Option entry/exit prices | `qd_get_option_price_over_time` | available — real traded bars, one call per contract-session |
| Contract choice | `strategy.md` §3 — ATM, third-Friday monthly | defined |
| Earnings veto | `earnings.py` | done |
| FOMC exposure | `fomc.py` | done, as evidence not veto |
| Capacity check | option-bar volume | available, must be enforced |
| Out-of-sample split | held back from the start | **not yet — decide before fitting anything** |

The order that matters: **fix the direction sign, then price with real option bars,
then test alignment as a gate.** Testing the gate first would measure it against
returns computed from an inverted signal.

One discipline point. The grid in section 5 is 9 combinations on 78 sessions, and the
alignment threshold adds more. That is enough surface to find something that looks
good by accident. Decide the out-of-sample split before running a single return
number, or the sample-size win from ~2,000 triggers gets spent on overfitting the
same way n=118 was.